# Baseline: Self-Attention (Transformer Encoder) for Unit Commitment Prediction

A second, deep-learning baseline for the same task as `baseline_random_forest.ipynb`, using a small
**non-causal (bidirectional) Transformer encoder** instead of a Random Forest.

Why non-causal rather than autoregressive: every exogenous input (day-ahead price, inflow) is known for
the *entire* 168-hour week in advance — there is no hidden future on the input side, so this is a
sequence-labeling problem, not a forecasting problem. The model attends over the whole known week at once
and predicts all 168x14 commitment decisions in a single parallel forward pass. An autoregressive decoder
that conditions each hour's decision on previous hours' *predicted* decisions (to capture minimum
up/down-time and start/stop-cost effects) is a plausible follow-up, deferred until this simpler model's
results motivate it.

The dataset is small for deep learning (2,922 labeled weekly cases, heavily overlapping daily rolling
windows), so the model and training setup below are deliberately small and regularized. The Random Forest
baseline's **0.62** validation micro-average ROC AUC is the number to beat.

## 1. Setup

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

from kaggle_metric import score as kaggle_score

HORIZON = 168  # hours in one weekly case

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("torch", torch.__version__, "| device:", device)

torch 2.14.1 | device: mps


## 2. Load data

Same loading helpers as `baseline_random_forest.ipynb`.

In [2]:
BASE_PATH = Path(".")
KERNEL_PATH = BASE_PATH / "data" / "kernel"
EXTENDED_PATH = BASE_PATH / "data" / "extended"


def load_series(path, date_col="date"):
    df = pd.read_csv(path, parse_dates=[date_col])
    df[date_col] = df[date_col].dt.tz_localize(None)
    return df


def series_arrays(df, date_col="date"):
    value_cols = [c for c in df.columns if c != date_col]
    return df[date_col].to_numpy(), df[value_cols].to_numpy(dtype=float), value_cols


price = load_series(KERNEL_PATH / "Historical_day_ahead_price_2015_2025.csv")
inflow = load_series(KERNEL_PATH / "Historical_inflow_1958_2025.csv")
volume = load_series(KERNEL_PATH / "Historical_volume_2015_2024.csv")
water_value = load_series(KERNEL_PATH / "Synthetic_water_value_2015_2024.csv")
min_flow = load_series(EXTENDED_PATH / "Constraint_min_flow.csv")
min_volume = load_series(EXTENDED_PATH / "Constraint_min_volume.csv")

uc = pd.read_csv(KERNEL_PATH / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
prediction_mapping = pd.read_csv(BASE_PATH / "prediction_mapping.csv", parse_dates=["starttime"])
sample_submission = pd.read_csv(BASE_PATH / "sample_submission.csv")

label_cols = [c for c in uc.columns if c.startswith("result_committed")]

price_dates, price_values, _ = series_arrays(price)
inflow_dates, inflow_values, inflow_cols = series_arrays(inflow)
volume_dates, volume_values, volume_cols = series_arrays(volume)
water_value_dates, water_value_values, water_value_cols = series_arrays(water_value)
min_flow_dates, min_flow_values, min_flow_cols = series_arrays(min_flow)
min_volume_dates, min_volume_values, min_volume_cols = series_arrays(min_volume)

N_DYNAMIC = 1 + len(inflow_cols)
N_STATIC = len(volume_cols) + len(water_value_cols) + len(min_flow_cols) + len(min_volume_cols)
print(f"{len(label_cols)} label columns, {len(uc)} labeled weekly cases, "
      f"{len(prediction_mapping)} cases to predict for submission")
print(f"dynamic features per hour: {N_DYNAMIC} (price + {len(inflow_cols)} inflow columns)")
print(f"static context features: {N_STATIC}")

2352 label columns, 2922 labeled weekly cases, 725 cases to predict for submission
dynamic features per hour: 19 (price + 18 inflow columns)
static context features: 43


## 3. Build sequence tensors

Unlike the Random Forest baseline (one flat feature vector per week), each case becomes a
`(168, N_DYNAMIC)` sequence of per-hour price + inflow, plus a `(N_STATIC,)` static context vector
(start-of-week volume / water value / min-flow / min-volume — identical start-of-week lookups to the RF
notebook's `build_features`, just kept separate instead of concatenated into one flat row).

The 2,352 label columns are **generator-major** (confirmed from the file header: all 168 `_t0..t167`
columns for one generator are contiguous before the next generator's block starts), so the generator name
order is derived programmatically from the column names rather than hardcoded, and a round-trip assertion
checks the reshape is correct before any training happens.

In [3]:
def build_sequence_features(starttimes):
    starttimes = pd.to_datetime(starttimes).to_numpy()

    price_pos = np.searchsorted(price_dates, starttimes)
    assert np.all(price_dates[price_pos] == starttimes), "starttime not aligned with price grid"
    idx168 = price_pos[:, None] + np.arange(HORIZON)[None, :]
    price_seq = price_values[idx168]  # (n, 168, 1)

    inflow_pos = np.searchsorted(inflow_dates, starttimes)
    assert np.all(inflow_dates[inflow_pos] == starttimes), "starttime not aligned with inflow grid"
    inflow_idx168 = inflow_pos[:, None] + np.arange(HORIZON)[None, :]
    inflow_seq = inflow_values[inflow_idx168]  # (n, 168, len(inflow_cols))

    dynamic_seq = np.concatenate([price_seq, inflow_seq], axis=-1).astype(np.float32)

    volume_feat = volume_values[np.searchsorted(volume_dates, starttimes)]
    water_value_feat = water_value_values[np.searchsorted(water_value_dates, starttimes)]
    min_flow_feat = min_flow_values[np.searchsorted(min_flow_dates, starttimes)]
    min_volume_feat = min_volume_values[np.searchsorted(min_volume_dates, starttimes)]
    static_ctx = np.hstack(
        [volume_feat, water_value_feat, min_flow_feat, min_volume_feat]
    ).astype(np.float32)

    return dynamic_seq, static_ctx


# Derive generator order from the label columns themselves (generator-major blocks of 168).
_pattern = re.compile(r"^result_committed_(.+)_t(\d+)$")
generator_names = []
_seen = set()
for c in label_cols:
    name = _pattern.match(c).group(1)
    if name not in _seen:
        _seen.add(name)
        generator_names.append(name)
n_generators = len(generator_names)
print(f"{n_generators} generators, in column order: {generator_names}")

uc_sorted = uc.sort_values("starttime").reset_index(drop=True)
dynamic_all, static_all = build_sequence_features(uc_sorted["starttime"])

Y_flat = uc_sorted[label_cols].to_numpy(dtype=np.float32)  # (N, 2352), generator-major
N = Y_flat.shape[0]
Y_all = Y_flat.reshape(N, n_generators, HORIZON).transpose(0, 2, 1)  # (N, 168, 14)

# Round-trip assertion: inverting the reshape must reproduce the original flat label layout exactly.
Y_roundtrip = Y_all.transpose(0, 2, 1).reshape(N, -1)
assert np.array_equal(Y_roundtrip, Y_flat), "label reshape does not round-trip correctly"

print("dynamic_all:", dynamic_all.shape, "static_all:", static_all.shape, "Y_all:", Y_all.shape)

pos_rates = pd.Series(Y_all.reshape(-1, n_generators).mean(axis=0), index=generator_names)
print("\nPer-generator commitment rate (fraction of hours ON):")
print(pos_rates.sort_values())

14 generators, in column order: ['Hogga_G1', 'Tokke_G1', 'Tokke_G2', 'Tokke_G3', 'Tokke_G4', 'Lio_G1', 'Byrte_G1', 'Vinje_G1', 'Vinje_G2', 'Vinje_G3', 'Haukeli_G1', 'Songa_G1', 'Kjela_G1', 'Vesle Kjela_G1']
dynamic_all: (2922, 168, 19) static_all: (2922, 43) Y_all: (2922, 168, 14)

Per-generator commitment rate (fraction of hours ON):
Vinje_G3          0.259149
Vinje_G2          0.420368
Vesle Kjela_G1    0.466883
Kjela_G1          0.470536
Tokke_G2          0.478354
Vinje_G1          0.485386
Songa_G1          0.485392
Tokke_G4          0.498195
Byrte_G1          0.542003
Haukeli_G1        0.546814
Tokke_G3          0.564262
Hogga_G1          0.594560
Tokke_G1          0.670991
Lio_G1            0.746533
dtype: float32


## 4. Time-based train / validation split

Identical logic to `baseline_random_forest.ipynb`: hold out the most recent ~15% of cases by
`starttime`, since cases are daily rolling 168h windows that overlap heavily and a random split would
leak near-duplicate windows across train/validation.

In [4]:
n_val = int(0.15 * len(uc_sorted))

dynamic_train, dynamic_val = dynamic_all[:-n_val], dynamic_all[-n_val:]
static_train, static_val = static_all[:-n_val], static_all[-n_val:]
Y_train, Y_val = Y_all[:-n_val], Y_all[-n_val:]
run_no_train = uc_sorted["Run No"].iloc[:-n_val].to_numpy()
run_no_val = uc_sorted["Run No"].iloc[-n_val:].to_numpy()

print(f"train: {dynamic_train.shape[0]} cases ({uc_sorted['starttime'].iloc[0].date()} to "
      f"{uc_sorted['starttime'].iloc[-n_val - 1].date()})")
print(f"val:   {dynamic_val.shape[0]} cases ({uc_sorted['starttime'].iloc[-n_val].date()} to "
      f"{uc_sorted['starttime'].iloc[-1].date()})")

train: 2484 cases (2015-01-01 to 2021-10-19)
val:   438 cases (2021-10-20 to 2022-12-31)


## 5. Normalize

Mean/std are fit on the **training split only** (to avoid leakage) and reused as-is for validation and,
later, for the unlabeled `prediction_mapping` cases.

In [5]:
dyn_mean = dynamic_train.reshape(-1, dynamic_train.shape[-1]).mean(axis=0)
dyn_std = dynamic_train.reshape(-1, dynamic_train.shape[-1]).std(axis=0) + 1e-6
static_mean = static_train.mean(axis=0)
static_std = static_train.std(axis=0) + 1e-6


def normalize(dynamic_seq, static_ctx):
    return (dynamic_seq - dyn_mean) / dyn_std, (static_ctx - static_mean) / static_std


dynamic_train_n, static_train_n = normalize(dynamic_train, static_train)
dynamic_val_n, static_val_n = normalize(dynamic_val, static_val)

## 6. Dataset / DataLoader

In [6]:
class UCDataset(torch.utils.data.Dataset):
    def __init__(self, dynamic_seq, static_ctx, target):
        self.dynamic_seq = torch.as_tensor(dynamic_seq, dtype=torch.float32)
        self.static_ctx = torch.as_tensor(static_ctx, dtype=torch.float32)
        self.target = torch.as_tensor(target, dtype=torch.float32)

    def __len__(self):
        return len(self.dynamic_seq)

    def __getitem__(self, idx):
        return self.dynamic_seq[idx], self.static_ctx[idx], self.target[idx]


train_dataset = UCDataset(dynamic_train_n, static_train_n, Y_train)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)

dynamic_val_t = torch.as_tensor(dynamic_val_n, dtype=torch.float32).to(device)
static_val_t = torch.as_tensor(static_val_n, dtype=torch.float32).to(device)

print(f"{len(train_dataset)} training sequences, batch size 32 -> {len(train_loader)} batches/epoch")

2484 training sequences, batch size 32 -> 78 batches/epoch


## 7. Model

A small Transformer encoder: project each hour's (price, inflow) into `d_model`, prepend one extra
token projected from the static start-of-week context, add a learned positional embedding, run through
a couple of non-causal (bidirectional) self-attention layers, then apply one shared `Linear(d_model, 14)`
head per hour to produce the 14 generator logits for that hour.

Kept deliberately small (`d_model=64`, 2 layers, 4 heads) given ~2,484 overlapping training sequences —
more capacity than this risks overfitting long before it helps.

In [7]:
class UCTransformer(nn.Module):
    def __init__(self, n_dynamic=N_DYNAMIC, n_static=N_STATIC, n_generators=n_generators,
                 horizon=HORIZON, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.2):
        super().__init__()
        self.input_proj = nn.Linear(n_dynamic, d_model)
        self.ctx_proj = nn.Linear(n_static, d_model)
        self.pos_embedding = nn.Parameter(torch.randn(horizon + 1, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_ff,
            dropout=dropout, batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.output_head = nn.Linear(d_model, n_generators)

    def forward(self, dynamic_seq, static_ctx):
        hourly_tokens = self.input_proj(dynamic_seq)          # (batch, 168, d_model)
        ctx_token = self.ctx_proj(static_ctx).unsqueeze(1)    # (batch, 1, d_model)
        tokens = torch.cat([ctx_token, hourly_tokens], dim=1)  # (batch, 169, d_model), no causal mask
        tokens = tokens + self.pos_embedding.unsqueeze(0)
        encoded = self.encoder(tokens)                        # (batch, 169, d_model)
        hourly_encoded = encoded[:, 1:, :]                    # drop the context-token position
        return self.output_head(hourly_encoded)                # (batch, 168, 14) logits


n_params = sum(p.numel() for p in UCTransformer().parameters())
print(f"UCTransformer parameter count: {n_params:,}")

UCTransformer parameter count: 82,766


## 8. Training loop

`BCEWithLogitsLoss` over all 168x14 outputs at once, `AdamW` with weight decay, and early stopping on
validation micro-average ROC AUC (via the actual `kaggle_metric.score`, not just loss) so the stopping
point is chosen on the metric that matters.

In [8]:
def predictions_to_submission_frame(proba, run_numbers):
    # proba: (n_cases, 168, 14) -> flatten back to the generator-major column layout of label_cols
    proba_flat = proba.transpose(0, 2, 1).reshape(proba.shape[0], -1)
    df = pd.DataFrame(proba_flat, columns=label_cols)
    df.insert(0, "Run No", run_numbers)
    return df


Y_val_flat = Y_val.transpose(0, 2, 1).reshape(Y_val.shape[0], -1)
solution_df = pd.DataFrame(Y_val_flat, columns=label_cols)
solution_df.insert(0, "Run No", run_no_val)


def compute_val_auc(model):
    model.eval()
    with torch.no_grad():
        proba = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()
    submission_df = predictions_to_submission_frame(proba, run_no_val)
    return kaggle_score(solution_df, submission_df)


model = UCTransformer().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

best_auc, best_epoch, best_state = -1.0, -1, None
patience, epochs_without_improvement, max_epochs = 15, 0, 150

start = time.time()
for epoch in range(max_epochs):
    model.train()
    train_losses = []
    for dyn, stat, tgt in train_loader:
        dyn, stat, tgt = dyn.to(device), stat.to(device), tgt.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(dyn, stat), tgt)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    val_auc = compute_val_auc(model)
    improved = val_auc > best_auc
    if improved:
        best_auc, best_epoch = val_auc, epoch
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epoch % 5 == 0 or improved:
        print(f"epoch {epoch:3d}  train_loss {np.mean(train_losses):.4f}  "
              f"val_auc {val_auc:.4f}  best_auc {best_auc:.4f}{' *' if improved else ''}")

    if epochs_without_improvement >= patience:
        print(f"early stopping at epoch {epoch} (best epoch {best_epoch}, val_auc {best_auc:.4f})")
        break

model.load_state_dict(best_state)
print(f"\nfit in {time.time() - start:.1f}s")
print(f"Best validation micro-average ROC AUC: {best_auc:.4f} (epoch {best_epoch}, RF baseline was ~0.62)")

epoch   0  train_loss 0.6567  val_auc 0.5607  best_auc 0.5607 *
epoch   1  train_loss 0.6084  val_auc 0.5898  best_auc 0.5898 *
epoch   2  train_loss 0.5681  val_auc 0.6112  best_auc 0.6112 *
epoch   3  train_loss 0.5433  val_auc 0.6323  best_auc 0.6323 *
epoch   4  train_loss 0.5241  val_auc 0.6504  best_auc 0.6504 *
epoch   5  train_loss 0.5088  val_auc 0.6783  best_auc 0.6783 *
epoch   6  train_loss 0.4944  val_auc 0.6866  best_auc 0.6866 *
epoch   7  train_loss 0.4848  val_auc 0.6917  best_auc 0.6917 *
epoch   8  train_loss 0.4723  val_auc 0.6991  best_auc 0.6991 *
epoch  10  train_loss 0.4538  val_auc 0.6946  best_auc 0.6991
epoch  11  train_loss 0.4477  val_auc 0.7001  best_auc 0.7001 *
epoch  12  train_loss 0.4430  val_auc 0.7054  best_auc 0.7054 *
epoch  15  train_loss 0.4256  val_auc 0.6973  best_auc 0.7054
epoch  20  train_loss 0.4032  val_auc 0.6919  best_auc 0.7054
epoch  25  train_loss 0.3909  val_auc 0.7015  best_auc 0.7054
early stopping at epoch 27 (best epoch 12, val_a

## 9. Per-generator breakdown

A diagnostic the RF notebook didn't compute: where does this model do well/poorly? This is also what
would motivate (or not) a follow-up autoregressive model targeting specific generators' temporal
dependencies.

In [9]:
model.eval()
with torch.no_grad():
    val_proba = torch.sigmoid(model(dynamic_val_t, static_val_t)).cpu().numpy()  # (n_val, 168, 14)

per_gen_auc = {}
for i, name in enumerate(generator_names):
    y_true = Y_val[:, :, i].reshape(-1)
    y_score = val_proba[:, :, i].reshape(-1)
    per_gen_auc[name] = roc_auc_score(y_true, y_score) if len(np.unique(y_true)) > 1 else float("nan")

pd.Series(per_gen_auc, name="val_auc").sort_values()

Haukeli_G1        0.551378
Songa_G1          0.622172
Tokke_G3          0.636473
Tokke_G4          0.639836
Vinje_G1          0.645296
Tokke_G2          0.646296
Vinje_G2          0.669258
Vinje_G3          0.691223
Tokke_G1          0.693698
Hogga_G1          0.723199
Vesle Kjela_G1    0.737909
Lio_G1            0.756561
Kjela_G1          0.802626
Byrte_G1          0.825775
Name: val_auc, dtype: float64

## 10. Build a submission for the unlabeled prediction cases

Refit on *all* labeled cases, using a fixed epoch budget taken from the best early-stopping epoch above
(rather than re-running early stopping with no held-out set left) — the same "refit on everything for
submission" pattern as the RF notebook, just with a fixed epoch count instead of a fixed estimator count.

In [10]:
dynamic_all_n, static_all_n = normalize(dynamic_all, static_all)
full_dataset = UCDataset(dynamic_all_n, static_all_n, Y_all)
full_loader = torch.utils.data.DataLoader(full_dataset, batch_size=32, shuffle=True)

final_model = UCTransformer().to(device)
final_optimizer = torch.optim.AdamW(final_model.parameters(), lr=3e-4, weight_decay=1e-2)
final_epochs = best_epoch + 1

start = time.time()
for epoch in range(final_epochs):
    final_model.train()
    for dyn, stat, tgt in full_loader:
        dyn, stat, tgt = dyn.to(device), stat.to(device), tgt.to(device)
        final_optimizer.zero_grad()
        loss = loss_fn(final_model(dyn, stat), tgt)
        loss.backward()
        final_optimizer.step()
print(f"refit on all {len(full_dataset)} labeled cases for {final_epochs} epochs in {time.time() - start:.1f}s")

refit on all 2922 labeled cases for 13 epochs in 20.9s


In [11]:
dynamic_submit, static_submit = build_sequence_features(prediction_mapping["starttime"])
dynamic_submit_n, static_submit_n = normalize(dynamic_submit, static_submit)

final_model.eval()
with torch.no_grad():
    dyn_t = torch.as_tensor(dynamic_submit_n, dtype=torch.float32).to(device)
    stat_t = torch.as_tensor(static_submit_n, dtype=torch.float32).to(device)
    submit_proba = torch.sigmoid(final_model(dyn_t, stat_t)).cpu().numpy()  # (n_submit, 168, 14)

submission = predictions_to_submission_frame(submit_proba, prediction_mapping["Run No"].to_numpy())
submission = submission[sample_submission.columns]  # match exact column order

assert submission.shape == sample_submission.shape
assert list(submission.columns) == list(sample_submission.columns)

submission.to_csv("transformer_submission.csv", index=False)
submission.head()

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6,result_committed_Hogga_G1_t7,result_committed_Hogga_G1_t8,...,result_committed_Vesle Kjela_G1_t158,result_committed_Vesle Kjela_G1_t159,result_committed_Vesle Kjela_G1_t160,result_committed_Vesle Kjela_G1_t161,result_committed_Vesle Kjela_G1_t162,result_committed_Vesle Kjela_G1_t163,result_committed_Vesle Kjela_G1_t164,result_committed_Vesle Kjela_G1_t165,result_committed_Vesle Kjela_G1_t166,result_committed_Vesle Kjela_G1_t167
0,2923,0.287258,0.238229,0.182879,0.144436,0.134619,0.129876,0.113917,0.137029,0.201293,...,0.915272,0.918881,0.923329,0.928851,0.905310,0.853511,0.574289,0.417851,0.234771,0.167283
1,2924,0.247627,0.242273,0.244718,0.255059,0.253022,0.282821,0.449977,0.527318,0.561482,...,0.912917,0.930528,0.942019,0.945351,0.930492,0.915058,0.883029,0.892443,0.736096,0.619358
2,2925,0.354007,0.343814,0.317612,0.306907,0.408500,0.525271,0.692909,0.735713,0.748496,...,0.980329,0.982075,0.981672,0.981938,0.981936,0.980901,0.976331,0.973243,0.955045,0.950830
3,2926,0.166671,0.126649,0.125389,0.130512,0.156024,0.180906,0.304084,0.395923,0.519788,...,0.981897,0.982032,0.981890,0.981195,0.974357,0.960775,0.924745,0.833829,0.387881,0.256965
4,2927,0.191930,0.189854,0.169591,0.204554,0.216463,0.272834,0.356584,0.372243,0.453970,...,0.954768,0.958478,0.967652,0.968683,0.962402,0.951382,0.910408,0.881253,0.591741,0.569200
